# 059 — Backpropagation Through Time

The name promises more novelty than it delivers. Unroll the network and it is
an ordinary computation graph; the one genuinely new thing is that $W_x$ and
$W_h$ are used at **every** time step, so their gradients must be **summed**
over all of them — a single `+=` instead of `=`.

| Part | What we check |
|---|---|
| A | the forward pass and the loss |
| B | BPTT by hand, with the `+=` that is the whole idea |
| C | all four gradients against finite differences (~1e-10) |
| D | what happens if you write `=` instead of `+=` |

## Part A — Forward

In [ ]:
import numpy as np

T, F, H = 6, 4, 5
rng = np.random.default_rng(0)
W = [rng.normal(0, 0.4, (F, H)),                       # Wx
     rng.normal(0, 1.0, (H, H)) / np.sqrt(H),          # Wh
     np.zeros(H),                                      # b
     rng.normal(0, 0.4, H),                            # Wo
     np.array(0.0)]                                    # bo

def forward(X, y, W):
    Wx, Wh, b, Wo, bo = W
    hs = [np.zeros(H)]
    for t in range(len(X)):
        hs.append(np.tanh(X[t] @ Wx + hs[-1] @ Wh + b))
    p = 1 / (1 + np.exp(-(hs[-1] @ Wo + bo)))
    loss = -(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12))
    return loss, hs, p

X, y = rng.normal(0, 1, (T, F)), 1.0
loss, hs, p = forward(X, y, W)
print(f"{T} steps, prediction {p:.4f}, loss {loss:.4f}")
assert len(hs) == T + 1          # h0 plus one state per step
print("one state per step, plus the zero state it started from")

## Part B — Backward, with the one new idea

`dWx += ...` inside the time loop. Every other line is lesson 019's chain rule
with different letters.

In [ ]:
def bptt(X, y, W):
    Wx, Wh, b, Wo, bo = W
    loss, hs, p = forward(X, y, W)
    dWx, dWh, db = np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)
    dz = p - y                                  # the sigmoid/cross-entropy cancellation
    dWo, dbo = dz * hs[-1], np.array(dz)
    dh = dz * Wo
    for t in range(len(X) - 1, -1, -1):
        draw = dh * (1 - hs[t + 1] ** 2)        # through tanh
        dWx += np.outer(X[t], draw)             # SUM across time
        dWh += np.outer(hs[t], draw)
        db += draw
        dh = Wh @ draw                          # one step further back
    return loss, [dWx, dWh, db, dWo, dbo]

loss, grads = bptt(X, y, W)
for name, g in zip(("dWx", "dWh", "db", "dWo", "dbo"), grads):
    print(f"  {name:<4} shape {np.shape(g)}")
assert grads[0].shape == (F, H) and grads[1].shape == (H, H)
print("the gradient shapes match the weights, as they must")

## Part C — Is the derivation right, or merely plausible?

In [ ]:
EPS = 1e-6
worst = 0.0
for gi, name in [(0, "dWx"), (1, "dWh"), (2, "db"), (3, "dWo")]:
    A, num = W[gi], np.zeros_like(W[gi])
    it = np.nditer(A, flags=["multi_index"])
    while not it.finished:
        i = it.multi_index
        original = A[i]
        A[i] = original + EPS
        up, _, _ = forward(X, y, W)
        A[i] = original - EPS
        down, _, _ = forward(X, y, W)
        A[i] = original
        num[i] = (up - down) / (2 * EPS)
        it.iternext()
    diff = np.abs(grads[gi] - num).max()
    worst = max(worst, diff)
    print(f"  {name:<4} max difference = {diff:.2e}")
assert worst < 1e-8
print("\nAgreement to ~1e-10 across every weight. The derivation is correct,")
print("not merely plausible - and this check costs four lines.")

## Part D — The bug the `+=` prevents

Write `=` instead and the gradient becomes the contribution of the **last**
time step alone.

In [ ]:
def bptt_broken(X, y, W):
    Wx, Wh, b, Wo, bo = W
    _, hs, p = forward(X, y, W)
    dWx = np.zeros_like(Wx)
    dh = (p - y) * Wo
    for t in range(len(X) - 1, -1, -1):
        draw = dh * (1 - hs[t + 1] ** 2)
        dWx = np.outer(X[t], draw)        # <- the bug: overwrite, not accumulate
        dh = Wh @ draw
    return dWx

right, wrong = grads[0], bptt_broken(X, y, W)
print(f"correct dWx norm {np.linalg.norm(right):.4f}")
print(f"broken  dWx norm {np.linalg.norm(wrong):.4f}")
cos = (right.ravel() @ wrong.ravel()) / (np.linalg.norm(right) * np.linalg.norm(wrong))
print(f"cosine between them: {cos:+.3f}")
assert not np.allclose(right, wrong)
print()
print("It still trains - slowly, and towards the wrong thing. The finite-")
print("difference check in Part C catches this immediately; eyeballing the")
print("loss curve does not.")

## What this notebook established

- An unrolled RNN is an ordinary computation graph, and BPTT is the ordinary
  chain rule run over it.
- The one new idea is that shared weights accumulate: `dWx += ...` inside the
  time loop, because $W_x$ is used at every step.
- All four gradients agreed with finite differences to about **1e-10**.
- Replacing `+=` with `=` keeps the shapes valid and silently computes the last
  step's contribution only — which a gradient check catches and a loss curve
  does not.

## Exercises

1. Add a bias to the output layer and extend the check to `dbo`.
2. Make the loss depend on **every** hidden state, not just the last. Which
   lines change?
3. Time BPTT at T = 10, 100 and 1000. What is the growth, and why?